# Multi-Lesion DR Staged Learning Pipeline -- Kaggle Notebook

This notebook runs the full pipeline on Kaggle with GPU acceleration.

## Prerequisites

1. **Enable GPU**: Settings -> Accelerator -> GPU T4 x2 (or P100)
2. **Add datasets** (click "+ Add Data" on the right sidebar):
   - [Diabetic Retinopathy Detection (EyePACS)](https://www.kaggle.com/c/diabetic-retinopathy-detection/data)
   - [DDR Dataset](https://www.kaggle.com/datasets/mariaherrerot/ddrdataset)
   - [IDRiD](https://www.kaggle.com/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset)
   - [e-ophtha](https://www.kaggle.com/datasets/nguyenhung1903/eophthaex) (optional, for external validation)
3. **Enable internet** if you need to clone the repo (Settings -> Internet -> On)

## 1. Setup and Installation

In [ ]:
# Clone the repo (only needed if not uploading as a dataset)
!git clone https://github.com/Ziad-Ahmed-Zaska/ECG-Image-Classification-Using-CNN.git repo
%cd repo
!git checkout feature/dr-multi-lesion-staged-pipeline

In [ ]:
# Install any missing dependencies (most are pre-installed on Kaggle)
!pip install -q opencv-python-headless scikit-learn

In [ ]:
# Add the repo root to Python path so dr_pipeline is importable
import sys, os

# If you cloned the repo:
REPO_ROOT = '/kaggle/working/repo'
# If you uploaded dr_pipeline as a Kaggle dataset:
# REPO_ROOT = '/kaggle/input/dr-pipeline'

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Verify import
import dr_pipeline
print(f'dr_pipeline version: {dr_pipeline.__version__}')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configure Dataset Paths

Update these paths to match where Kaggle mounts your datasets.
Kaggle datasets are typically at `/kaggle/input/<dataset-name>/`.

In [ ]:
# ============================================================
# UPDATE THESE PATHS to match your Kaggle dataset names
# ============================================================

# EyePACS -- from the Diabetic Retinopathy Detection competition
EYEPACS_ROOT = '/kaggle/input/diabetic-retinopathy-detection'
EYEPACS_TRAIN_IMAGES = os.path.join(EYEPACS_ROOT, 'train')
EYEPACS_LABELS_CSV = os.path.join(EYEPACS_ROOT, 'trainLabels.csv')

# DDR
DDR_ROOT = '/kaggle/input/ddrdataset'

# IDRiD
IDRID_ROOT = '/kaggle/input/indian-diabetic-retinopathy-image-dataset'

# e-ophtha (optional)
EOPHTHA_ROOT = '/kaggle/input/eophthaex'

# Output directory (Kaggle allows writing to /kaggle/working/)
OUTPUT_DIR = '/kaggle/working/dr_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Dataset paths configured.')
for name, path in [('EyePACS', EYEPACS_ROOT), ('DDR', DDR_ROOT), ('IDRiD', IDRID_ROOT)]:
    exists = os.path.isdir(path)
    print(f'  {name}: {path} -- {"FOUND" if exists else "NOT FOUND (add this dataset)"}')

## 3. Prepare Dataset Metadata CSVs

The pipeline expects a standardised metadata CSV per dataset.
This section converts each dataset's native format into the expected schema.

In [ ]:
import pandas as pd
import glob

# ---- EyePACS metadata ----
# The competition CSV has columns: image, level (0-4)
if os.path.isfile(EYEPACS_LABELS_CSV):
    df_eyepacs = pd.read_csv(EYEPACS_LABELS_CSV)
    grade_map = {0: 'no_dr', 1: 'mild', 2: 'moderate', 3: 'severe', 4: 'proliferative'}
    df_eyepacs['label'] = df_eyepacs['level'].map(grade_map)
    # Patient ID: strip the _left / _right suffix
    df_eyepacs['patient_id'] = df_eyepacs['image'].str.replace(r'_(left|right)$', '', regex=True)
    # Build full image path
    df_eyepacs['image_path'] = df_eyepacs['image'] + '.jpeg'
    
    eyepacs_meta_path = os.path.join(OUTPUT_DIR, 'eyepacs_metadata.csv')
    df_eyepacs[['image_path', 'patient_id', 'label']].to_csv(eyepacs_meta_path, index=False)
    print(f'EyePACS metadata: {len(df_eyepacs)} images, saved to {eyepacs_meta_path}')
    print(df_eyepacs['label'].value_counts())
else:
    print('EyePACS labels CSV not found. Skipping.')
    eyepacs_meta_path = None

In [ ]:
# ---- DDR metadata ----
# DDR typically has train/valid/test folders and label files.
# Adjust this cell based on the exact DDR dataset version you added.

if os.path.isdir(DDR_ROOT):
    # List what's actually in the DDR dataset directory
    print('DDR contents:', os.listdir(DDR_ROOT))
    
    # Example: build metadata from DDR's lesion annotation files
    # This is a template -- adjust column names to match your DDR version
    ddr_records = []
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(DDR_ROOT, split, 'image')
        label_file = os.path.join(DDR_ROOT, split, 'label.txt')
        if os.path.isdir(img_dir) and os.path.isfile(label_file):
            with open(label_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        fname = parts[0]
                        ddr_records.append({
                            'image_path': os.path.join(split, 'image', fname),
                            'patient_id': fname.split('.')[0],
                            'microaneurysm': 0, 'haemorrhage': 0,
                            'hard_exudate': 0, 'soft_exudate': 0,
                        })
    
    if ddr_records:
        df_ddr = pd.DataFrame(ddr_records)
        ddr_meta_path = os.path.join(OUTPUT_DIR, 'ddr_metadata.csv')
        df_ddr.to_csv(ddr_meta_path, index=False)
        print(f'DDR metadata: {len(df_ddr)} images, saved to {ddr_meta_path}')
    else:
        print('Could not parse DDR structure. Check the directory layout and adjust this cell.')
        ddr_meta_path = None
else:
    print('DDR dataset not found. Skipping.')
    ddr_meta_path = None

## 4. Build Dataset Manifests (Patient-Level Splits)

In [ ]:
from dr_pipeline.datasets.dataset_manifest import DatasetManifest

# EyePACS manifest
if eyepacs_meta_path:
    eyepacs_manifest = DatasetManifest(
        dataset_name='eyepacs',
        root_dir=EYEPACS_TRAIN_IMAGES,
        train_ratio=0.70,
        val_ratio=0.15,
    )
    eyepacs_manifest.load_metadata(eyepacs_meta_path)
    eyepacs_manifest.build_patient_splits()
    eyepacs_manifest.save(os.path.join(OUTPUT_DIR, 'eyepacs_manifest'))
    
    for s in eyepacs_manifest.summary():
        print(f'  {s.split}: {s.num_patients} patients, {s.num_images} images')
else:
    eyepacs_manifest = None
    print('Skipping EyePACS manifest (no metadata).')

## 5. Stage 2: Encoder Pretraining on EyePACS

Pick one variant:
- **A**: Self-supervised only (SimCLR) -- best if labels are noisy
- **B**: Supervised DR grading only
- **C**: SSL then supervised fine-tune (recommended)

In [ ]:
from dr_pipeline.config import TrainConfig, CHECKPOINT_DIR, ensure_dirs
from dr_pipeline.models.encoder import DREncoder
from dr_pipeline.models.self_supervised import SimCLRPretrainer
from dr_pipeline.models.dr_grading import DRGradingModel
from dr_pipeline.datasets.eyepacs import EyePACSDataset
from torch.utils.data import DataLoader
import torch

# Override output paths for Kaggle
import dr_pipeline.config as cfg
from pathlib import Path
cfg.OUTPUT_DIR = Path(OUTPUT_DIR)
cfg.CHECKPOINT_DIR = Path(OUTPUT_DIR) / 'checkpoints'
cfg.LOG_DIR = Path(OUTPUT_DIR) / 'logs'
cfg.FIGURE_DIR = Path(OUTPUT_DIR) / 'figures'
ensure_dirs()

VARIANT = 'B'  # Change to 'A', 'B', or 'C'

train_cfg = TrainConfig(
    batch_size=32,       # reduce if you hit OOM
    learning_rate=1e-4,
    epochs=30,           # increase for real training
    ssl_epochs=50,       # for variant A/C
    encoder_name='resnet50',
    mixed_precision=True,
)

print(f'Training config: variant={VARIANT}, batch_size={train_cfg.batch_size}, encoder={train_cfg.encoder_name}')

In [ ]:
# Build data loaders
if eyepacs_manifest is not None:
    train_records = eyepacs_manifest.get_split('train')
    val_records = eyepacs_manifest.get_split('val')

    if VARIANT in ('A', 'C'):
        ssl_dataset = EyePACSDataset(train_records, root_dir=EYEPACS_TRAIN_IMAGES, mode='ssl')
        ssl_loader = DataLoader(ssl_dataset, batch_size=train_cfg.batch_size,
                                shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
        print(f'SSL dataset: {len(ssl_dataset)} images')

    if VARIANT in ('B', 'C'):
        train_ds = EyePACSDataset(train_records, root_dir=EYEPACS_TRAIN_IMAGES, mode='supervised')
        val_ds = EyePACSDataset(val_records, root_dir=EYEPACS_TRAIN_IMAGES, mode='supervised')
        train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size,
                                  shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=train_cfg.batch_size,
                                shuffle=False, num_workers=2, pin_memory=True)
        print(f'Supervised dataset: {len(train_ds)} train, {len(val_ds)} val')
else:
    print('No EyePACS data available. Using ImageNet-pretrained encoder.')

In [ ]:
# Run pretraining
from dr_pipeline.training.train_pretrain import train_ssl, train_supervised

encoder = None

if eyepacs_manifest is not None:
    if VARIANT in ('A', 'C'):
        print('=== Stage 2A: Self-supervised pretraining ===')
        encoder = train_ssl(train_cfg, ssl_loader)
        print('SSL pretraining complete.')
    
    if VARIANT in ('B', 'C'):
        print('=== Stage 2B: Supervised DR grading ===')
        encoder = train_supervised(train_cfg, train_loader, val_loader, encoder=encoder)
        print('Supervised DR grading complete.')
else:
    # Fall back to ImageNet-pretrained encoder
    encoder = DREncoder(backbone_name=train_cfg.encoder_name, pretrained_imagenet=True)
    print('Using ImageNet-pretrained encoder (no EyePACS data).')

print(f'Encoder feature dim: {encoder.get_feat_dim()}')

## 6. Stage 3: Multi-Lesion Classification on DDR

In [ ]:
from dr_pipeline.datasets.ddr import DDRDataset
from dr_pipeline.training.train_lesion_classifier import train_lesion_classifier

USE_MULTI_SCALE = True  # Set to True for the patch MIL branch

if ddr_meta_path is not None:
    ddr_manifest = DatasetManifest(dataset_name='ddr', root_dir=DDR_ROOT)
    ddr_manifest.load_metadata(ddr_meta_path)
    ddr_manifest.build_patient_splits()
    
    ddr_train = DDRDataset(ddr_manifest.get_split('train'), root_dir=DDR_ROOT,
                           return_patches=USE_MULTI_SCALE)
    ddr_val = DDRDataset(ddr_manifest.get_split('val'), root_dir=DDR_ROOT,
                         return_patches=USE_MULTI_SCALE)
    
    ddr_train_loader = DataLoader(ddr_train, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
    ddr_val_loader = DataLoader(ddr_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
    
    lesion_cfg = TrainConfig(batch_size=16, learning_rate=1e-4, epochs=30)
    
    print(f'=== Stage 3: Lesion classification (multi_scale={USE_MULTI_SCALE}) ===')
    lesion_model, thresholds = train_lesion_classifier(
        lesion_cfg, ddr_train_loader, ddr_val_loader,
        encoder=encoder, multi_scale=USE_MULTI_SCALE
    )
    print(f'Optimal thresholds: {thresholds.tolist()}')
else:
    print('DDR dataset not available. Skipping Stage 3.')
    lesion_model = None

## 7. Stage 4: Explainability and Weak Localization

In [ ]:
from dr_pipeline.evaluation.explainability import GradCAM, generate_attributions, pointing_game_accuracy
from dr_pipeline.utils.visualization import build_attribution_montage
from dr_pipeline.config import LESION_CLASSES

if lesion_model is not None:
    print('=== Stage 4: Generating attribution maps ===')
    
    # Get a small batch for visualization
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    lesion_model.to(device).eval()
    
    # Take first batch from val set for qualitative panel
    for batch in ddr_val_loader:
        if USE_MULTI_SCALE:
            sample_images = batch[0][:4].to(device)
        else:
            sample_images = batch[0][:4].to(device)
        break
    
    # Generate Grad-CAM attributions
    attributions = generate_attributions(
        lesion_model, sample_images, method='gradcam'
    )
    
    for cls_name in LESION_CLASSES:
        if cls_name in attributions:
            print(f'  {cls_name}: {len(attributions[cls_name])} heatmaps generated')
    
    print('Attribution maps generated. See montage in output.')
else:
    print('Skipping Stage 4 (no lesion model).')

## 8. Stage 5: Lesion Segmentation on IDRiD

In [ ]:
from dr_pipeline.datasets.idrid import IDRiDDataset
from dr_pipeline.training.train_segmentation import train_segmentation, compute_seg_metrics

if os.path.isdir(IDRID_ROOT):
    print('=== Stage 5: Segmentation training ===')
    print(f'IDRiD contents: {os.listdir(IDRID_ROOT)}')
    
    # You need to create IDRiD metadata CSV similar to the DDR cell above.
    # This is dataset-specific; adjust paths based on the actual IDRiD layout.
    
    # Example placeholder:
    # idrid_manifest = DatasetManifest(dataset_name='idrid', root_dir=IDRID_ROOT)
    # idrid_manifest.load_metadata(idrid_meta_path)
    # idrid_manifest.build_patient_splits()
    # ...
    
    print('IDRiD metadata preparation needed -- adjust the cell above for your dataset layout.')
else:
    print('IDRiD dataset not found. Skipping Stage 5.')

## 9. Evaluation Report

In [ ]:
from dr_pipeline.evaluation.metrics import classification_report
from dr_pipeline.evaluation.calibration import expected_calibration_error
import numpy as np
import json

if lesion_model is not None and ddr_meta_path is not None:
    print('=== Evaluation Report ===')
    
    # Collect predictions on test set
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    lesion_model.to(device).eval()
    
    ddr_test = DDRDataset(ddr_manifest.get_split('test'), root_dir=DDR_ROOT,
                          return_patches=False)
    test_loader = DataLoader(ddr_test, batch_size=16, shuffle=False, num_workers=2)
    
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            logits = lesion_model(images)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    
    # Full report
    report = classification_report(all_labels, all_probs, thresholds.numpy())
    print(json.dumps(report, indent=2, default=str))
    
    # Calibration
    ece, _, _, _ = expected_calibration_error(all_labels, all_probs)
    print(f'\nExpected Calibration Error (ECE): {ece:.4f}')
    
    # Save report
    report_path = os.path.join(OUTPUT_DIR, 'evaluation_report.json')
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2, default=str)
    print(f'Report saved to {report_path}')
else:
    print('No model or data available for evaluation.')

## 10. Run Unit Tests (Sanity Check)

In [ ]:
# Run the test suite to verify everything works
!cd {REPO_ROOT} && python -m pytest tests/test_dr_pipeline.py -v --tb=short 2>&1 | tail -30

## Tips for Kaggle

1. **Memory**: If you hit OOM, reduce `batch_size` (try 8 or 16) or use `resnet18` instead of `resnet50`.
2. **Training time**: Stage 2 SSL on full EyePACS can take hours. Start with a subset:
   ```python
   train_records = train_records[:5000]  # quick experiment
   ```
3. **Save checkpoints**: Kaggle kills notebooks after ~12h. Save checkpoints to `/kaggle/working/` and resume.
4. **Dataset versions**: Different Kaggle uploads of DDR/IDRiD may have different directory layouts. Print `os.listdir()` to inspect.
5. **Reproducibility**: The pipeline uses `seed=42` by default for all splits and random operations.